In [1]:
from dotenv import load_dotenv
import os

# .env 파일 로드 (override=False가 기본값이므로 시스템 환경변수가 우선순위를 가집니다)
load_dotenv()

# 정상 로드 확인 테스트
print("OpenAI 키:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키:", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

OpenAI 키: sk-proj-...
LANGSMITH 키: lsv2_pt_...
LangSmith 프로젝트: langchain-study


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

from langchain_teddynote import logging
logging.langsmith(os.getenv("LANGSMITH_PROJECT"), set_enable=True)

model = ChatOpenAI(model="gpt-4o-mini",max_completion_tokens=2048).bind(logprobs=True)

LangSmith 추적을 시작합니다.
[프로젝트명]
langchain-study


In [3]:
chain_1 = (
    PromptTemplate.from_template("{country}의 수도는 어디야?") |
    model |
    StrOutputParser()
)

chain_2 = (
    PromptTemplate.from_template("{country}의 면적은 얼마야?") |
    model |
    StrOutputParser()
)

print(chain_1)
print(chain_2)

first=PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디야?') middle=[_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001E0FA05C4D0>, async_client

In [4]:
combined = RunnableParallel(capital = chain_1, area = chain_2)
print(combined)

steps__={'capital': PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디야?')
| _ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001E0FA05C4D0>, asyn

In [5]:
c1 = chain_1.invoke({"country" : "이스라엘"})
c2 = chain_2.invoke({"country" : "아랍"})
c3 = combined.invoke({"country" : "베네수엘라"})

print(c1)
print(c2)
print(c3)


이스라엘의 수도는 예루살렘입니다. 하지만 국제 사회에서는 예루살렘의 지위에 대해 여러 가지 의견이 존재하며, 일부 국가는 텔아비브를 이스라엘의 수도로 인정하기도 합니다. 이 문제는 복잡하고 정치적인 논란이 많습니다.
아랍 세계(아랍 국가들이 포함된 지역)의 면적은 대략 14.5 백만 제곱킬로미터(약 5.6 백만 제곱마일)로 추정됩니다. 이는 북아프리카에서 아라비아 반도에 이르는 여러 국가들이 포함된 면적입니다. 아랍 국가들의 면적은 서로 다르지만, 이 지역의 총 면적은 상당히 광범위합니다.
{'capital': '베네수엘라의 수도는 카라카스(Caracas)입니다.', 'area': '베네수엘라의 면적은 약 916,445 평방킬로미터입니다. 이는 남아메리카에서 가장 큰 국가 중 하나로, 다양한 지형과 생태계를 가지고 있습니다.'}


In [7]:
b1 = chain_1.batch([{"country" : "브라질"}, {"country" : "몽골"}])

b2 = chain_2.batch([{"country" : "브라질"}, {"country" : "몽골"}])

b3 = combined.batch([{"country" : "브라질"}, {"country" : "몽골"}])

print(b1)
print(b2)
print(b3)



['브라질의 수도는 브라질리아입니다. 브라질리아는 1960년에 공식적으로 수도로 지정되었으며, 국가의 중앙부에 위치하고 있습니다.', '몽골의 수도는 울란바토르(Ulaanbaatar)입니다. 울란바토르는 몽골의 정치, 경제, 문화의 중심지로, 나라의 대다수 인구가 이곳에 살고 있습니다.']
['브라질의 면적은 약 8,515,767 평방킬로미터(약 3,287,956 평방 마일)입니다. 이는 세계에서 다섯 번째로 큰 나라입니다.', '몽골의 면적은 약 1,564,116 평방킬로미터입니다. 이는 세계에서 18번째로 큰 국가에 해당합니다. 몽골은 광활한 초원과 산악 지형으로 유명하며, 인구 밀도가 낮은 편입니다.']
[{'capital': '브라질의 수도는 브라지리아입니다. 브라지리아는 1960년에 수도로 지정되었으며, 현대적인 도시 설계로 유명합니다.', 'area': '브라질의 면적은 약 8,515,767 평방킬로미터입니다. 이는 브라질이 세계에서 다섯 번째로 큰 나라임을 의미합니다.'}, {'capital': '몽골의 수도는 울란바토르(Улаанбаатар)입니다. 울란바토르는 몽골의 정치, 경제, 문화 중심지로, 몽골의 대부분 인구가 이 도시에 거주하고 있습니다.', 'area': '몽골의 면적은 약 1,564,116 평방킬로미터로, 세계에서 18위의 큰 나라입니다. 이는 몽골이 북한과 남한을 합한 것보다도 큰 면적입니다.'}]


In [8]:
from langchain_core.runnables import RunnablePassthrough

prompt = PromptTemplate.from_template("{num}의 10배는?")

#Create Chain
chain = prompt | model
print(chain)


first=PromptTemplate(input_variables=['num'], input_types={}, partial_variables={}, template='{num}의 10배는?') middle=[] last=_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001E0FA05C4D0>, async_client=<op

In [9]:
chain.invoke({"num": 5})

AIMessage(content='5의 10배는 50입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 14, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_de64dd5ae6', 'id': 'chatcmpl-ENvWofRaTnQOKkbPF0PkSfJoN7HfG', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': {'content': [{'token': '5', 'bytes': [53], 'logprob': 0.0, 'top_logprobs': []}, {'token': '의', 'bytes': [236, 157, 152], 'logprob': -9.088346359931165e-07, 'top_logprobs': []}, {'token': ' ', 'bytes': [32], 'logprob': 0.0, 'top_logprobs': []}, {'token': '10', 'bytes': [49, 48], 'logprob': 0.0, 'top_logprobs'

In [10]:
chain.invoke(10)

AIMessage(content='10의 10배는 100입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 14, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eac509c072', 'id': 'chatcmpl-ENvX87tFdAJdA9mDwhNWJzb5wWnWE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': {'content': [{'token': '10', 'bytes': [49, 48], 'logprob': 0.0, 'top_logprobs': []}, {'token': '의', 'bytes': [236, 157, 152], 'logprob': -5.512236498361744e-07, 'top_logprobs': []}, {'token': ' ', 'bytes': [32], 'logprob': 0.0, 'top_logprobs': []}, {'token': '10', 'bytes': [49, 48], 'logprob': 0.0, 'top_lo

In [11]:
RunnablePassthrough().invoke({"num": 11})

{'num': 11}

In [13]:
num_1 = chain.invoke({"num": 5})

num_2 = chain.invoke(5)

num_3 = RunnablePassthrough().invoke({"num": 10})

print(num_1)
print(num_2)
print(num_3)

content='5의 10배는 50입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 14, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_de64dd5ae6', 'id': 'chatcmpl-ENvZQB5V6d0L6GedwIhiBEoIrI8OQ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': {'content': [{'token': '5', 'bytes': [53], 'logprob': 0.0, 'top_logprobs': []}, {'token': '의', 'bytes': [236, 157, 152], 'logprob': -7.896309739408025e-07, 'top_logprobs': []}, {'token': ' ', 'bytes': [32], 'logprob': 0.0, 'top_logprobs': []}, {'token': '10', 'bytes': [49, 48], 'logprob': 0.0, 'top_logprobs': []}, {'tok

In [14]:
runnable_chain = {"num" : RunnablePassthrough()} | prompt | ChatOpenAI()
runnable_chain.invoke(13)

AIMessage(content='130입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 3, 'prompt_tokens': 15, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ENvao3Qa3p2q6hUVLobLI4wyRiWel', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09edc-60ec-76b2-be47-09fe72768eb0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 3, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [19]:
(RunnablePassthrough.assign(new_num = lambda x:x["num"] * 7)).invoke({"num" : 3}) 

{'num': 3, 'new_num': 21}

In [21]:
from langchain_core.runnables import RunnableParallel

runnable = RunnableParallel(
    passed = RunnablePassthrough(),
    extra = RunnablePassthrough.assign(mult = lambda x:x["num"] * 3),
    modified = lambda x:x["num"] + 1,
)

runnable.invoke({"num":3})

{'passed': {'num': 3}, 'extra': {'num': 3, 'mult': 9}, 'modified': 4}

In [22]:
chain_1 = (
    {"country" : RunnablePassthrough()}
    | PromptTemplate.from_template("{country}의 수도는?")
    | model
)

chain_2 = (
    {"country": RunnablePassthrough()}
    | PromptTemplate.from_template("{country}의 면적은?")
    | model
)

In [23]:
combined_chain = RunnableParallel(capital = chain_1, area = chain_2)
combined_chain.invoke("이집트")


{'capital': AIMessage(content='이집트의 수도는 카이로(Cairo)입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 14, 'total_tokens': 28, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8c8cb050a6', 'id': 'chatcmpl-ENvoDA9TXkr9XOf7TQ1NemnKj7iEN', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': {'content': [{'token': '이', 'bytes': [236, 157, 180], 'logprob': -1.5213274309644476e-05, 'top_logprobs': []}, {'token': '집', 'bytes': [236, 167, 145], 'logprob': 0.0, 'top_logprobs': []}, {'token': '트', 'bytes': [237, 138, 184], 'logprob': -5.512236498361744e-07, 'top_logprobs': []}, {'

### 함수를 실행하는 RunableLambda

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from datetime import datetime

#RunanableLambda에서 invoke로 함수가 호출될때 오류가 발생할 수 있기 때문에 임의의 매개변수 지정
# a -> _ 표기가 가독성이 좋을 것으로 보인다.
#필요하다면 lambda식 호출 과정에서 RunnableLambda(lambda _ : get_today())로 호출한다면 매개변수를 않넣어도 된다.
def get_today(a): 
    return datetime.today().strftime("%b-%d")

print(get_today(None))

Sep-14


In [29]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

prompt = PromptTemplate.from_template(
    "{today}가 생일인 유명인이나 가수 {n}명을 나열 하세요. 생년월일을 표기해 주세요."
)
model = ChatOpenAI(temperature=0, model="gpt-4o-mini")

chain = (
    {"today": RunnableLambda(get_today), "n": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()    
)

In [30]:
print(chain.invoke(3))

다음은 9월 14일에 생일인 유명인과 가수 3명입니다:

1. **안젤리나 졸리 (Angelina Jolie)** - 1975년 9월 14일
2. **매리 블랙 (Mary Black)** - 1955년 9월 14일
3. **제이슨 스태덤 (Jason Statham)** - 1967년 9월 14일

이 외에도 많은 유명인들이 이 날에 태어났습니다!


In [38]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI

def length_function(text):
    return len(text)

def _multiple_length_function(text_1, text_2):
    return len(text_1) * len(text_2)

def multiple_length_function(_dict):
    return _multiple_length_function(_dict["text_1"], _dict["text_2"])

prompt = ChatPromptTemplate.from_template("{a} + {b}는 무엇인가요?")
model = ChatOpenAI()
chain = prompt | model

chain = (
    {
        "a" : itemgetter("word_1") | RunnableLambda(length_function),
        "b" : {"text_1" : itemgetter("word_1"), "text_2" : itemgetter("word_2")} 
        | RunnableLambda(multiple_length_function),
    }
    | prompt
    | model 
)

In [40]:
answer = chain.invoke({"word_1": "hello,", "word_2": "world!"})
print(answer.content)

6 + 36 = 42입니다.
